# Lesson 3.3 - Model Evaluation & Selection

## Objectives
- Compare models with stratified cross-validation.
- Tune hyperparameters with grid search.
- Select model using business-aligned metrics and leakage-safe pipelines.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier


## Section A - Split and Leakage-Safe Pipelines


In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

models = {
    "log_reg": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
    "random_forest": RandomForestClassifier(n_estimators=250, random_state=42),
    "grad_boost": GradientBoostingClassifier(random_state=42),
    "xgboost": XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=42),
}


## Section B - Cross-Validation Comparison


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"auc": "roc_auc", "f1": "f1", "acc": "accuracy"}
rows = []
for name, model in models.items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, "cv_auc": s["test_auc"].mean(), "cv_f1": s["test_f1"].mean(), "cv_acc": s["test_acc"].mean()})

cv_df = pd.DataFrame(rows).sort_values("cv_auc", ascending=False)
display(cv_df)


## Section C - Grid Search Example


In [ ]:
rf_grid = {"n_estimators": [150, 250], "max_depth": [None, 5, 10], "min_samples_split": [2, 5]}
rf_search = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
rf_search.fit(X_train, y_train)
print(rf_search.best_params_)
print(rf_search.best_score_)


## Section D - Final Test Evaluation


In [ ]:
best_models = {
    "log_reg": models["log_reg"],
    "rf_tuned": rf_search.best_estimator_,
    "grad_boost": models["grad_boost"],
    "xgboost": models["xgboost"],
}
rows = []
for name, model in best_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({"model": name, "test_auc": roc_auc_score(y_test, proba), "test_f1": f1_score(y_test, pred), "test_acc": accuracy_score(y_test, pred)})

final_df = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
display(final_df)

best_name = final_df.iloc[0]["model"]
best_model = best_models[best_name]
print("best:", best_name)
print(confusion_matrix(y_test, best_model.predict(X_test)))


In [ ]:
def assert_selection_quality(df, min_auc=0.95):
    if df["test_auc"].max() < min_auc:
        raise ValueError(f"No model met minimum AUC {min_auc}")

assert_selection_quality(final_df)
print("quality gate passed")


## Business Case Studies & Exceptions
- Highest AUC model may fail latency/compliance constraints.
- Leakage from global preprocessing can fake strong metrics.
- Use CV variance + threshold gates before promotion.


## Interview Questions & Answers
1. **Q:** Why separate train/val/test?  
   **A:** Avoid optimistic bias during model tuning.
2. **Q:** What is cross-validation?  
   **A:** Repeated fold-based performance estimation.
3. **Q:** Why stratified CV?  
   **A:** Preserve class balance per fold.
4. **Q:** Grid vs random search?  
   **A:** Grid exhaustive; random often more efficient.
5. **Q:** What is leakage?  
   **A:** Training uses information unavailable at inference.
6. **Q:** Why pipeline for preprocessing?  
   **A:** Keep fit/transform leak-safe in CV.
7. **Q:** Why not accuracy only?  
   **A:** Misses class-cost tradeoffs.
8. **Q:** How evaluate imbalanced datasets?  
   **A:** Use precision/recall/F1/PR-AUC.
9. **Q:** CV high but test low means?  
   **A:** Possible overfit or data shift.
10. **Q:** Model choice beyond metrics?  
    **A:** Latency, cost, explainability, governance.
